In [ ]:
import os
import json
import time
import math
import hashlib
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer
from FlagEmbedding import FlagReranker
import faiss

from openai import OpenAI

# ---- Paths (edit if needed) ----
DATA_PATH = Path("../corpus/unified_training_dataset.jsonl")
EMB_MODEL_PATH = Path("../BioLORD-2023")                 # Embedding model
RERANK_MODEL_PATH = Path("../models/bge-reranker-v2-m3")      # Reranking model
MODEL_PATH = "/root/autodl-tmp/deberta-large-mnli" # <<-- NLI model

OUT_DIR = Path("./MedEmbedding"); OUT_DIR.mkdir(exist_ok=True, parents=True)
out_path = "./MedEmbedding/diagnostics.csv" # <<-- CSV file dir

# ---- DeepSeek/OpenAI-compatible API ----
DEEPSEEK_BASE_URL = os.environ.get("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    print("Please set environment variable DEEPSEEK_API_KEY before running generation cells.")

# ---- Device ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Please set environment variable DEEPSEEK_API_KEY before running generation cells.
Device: cuda


In [3]:
TOP_K = 30            # retrieval depth
TOP_K_RERANK = 8      # reranker depth
MAX_SAMPLES = 0       # 0 = run all
MODEL_NAME = "deepseek-chat"  # or deepseek-reasoner

In [ ]:
import os
from openai import OpenAI

API_KEY = ""   # <-- DeepSeek API Key

os.environ["DEEPSEEK_API_KEY"] = API_KEY
os.environ["OPENAI_API_KEY"] = API_KEY  

# DeepSeek base url
DEEPSEEK_BASE_URL = "https://api.deepseek.com"

# 初始化 client
client = OpenAI(api_key=API_KEY, base_url=DEEPSEEK_BASE_URL)

print("DeepSeek client configured. Key prefix:", API_KEY[:10], "...")

DeepSeek client configured. Key prefix: sk-832ba59 ...


In [5]:
# Utility helpers

def md5_8(s: Any) -> str:
    """Short MD5 hash for stable ids."""
    return hashlib.md5(str(s).encode("utf-8")).hexdigest()[:8]

def read_jsonl(path: Path):
    """Yield JSON objects line by line."""
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def write_jsonl(path: Path, rows: List[Dict[str, Any]]):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def normalize_label(raw: Optional[str]) -> Optional[str]:
    """Normalize labels to {Support, Refute, Neutral} if possible."""
    if raw is None:
        return None
    t = str(raw).strip().lower()
    if t in ["support", "supports", "true", "supporting"]:
        return "Support"
    if t in ["refute", "refutes", "false", "contradict", "contradictory", "contradicting"]:
        return "Refute"
    if t in ["neutral", "mixture", "unproven", "snopes"]:
        return "Neutral"
    return None

In [6]:
# Build a unique evidence corpus and a claim list from the unified JSONL.

data = list(read_jsonl(DATA_PATH))
print(f"Loaded {len(data)} unified examples")

# Build corpus (unique evidence texts)
# We deduplicate by normalized text to reduce index size.
corpus_texts: List[str] = []
corpus_meta:  List[Dict[str, Any]] = []   # keep optional metadata for each evidence
text2idx: Dict[str, int] = {}

def norm_text(s: str) -> str:
    return " ".join(s.split()).strip()

for ex in data:
    evs = ex.get("evidence", []) or []
    for ev in evs:
        txt = str(ev.get("text", "") or "")
        if not txt.strip():
            continue
        key = norm_text(txt)
        if key not in text2idx:
            text2idx[key] = len(corpus_texts)
            corpus_texts.append(key)
            corpus_meta.append({
                "source": (ex.get("meta") or {}).get("source"),
                "doc_id": ev.get("doc_id"),
                "from_example_id": ex.get("id")
            })

print(f"Unique evidence texts: {len(corpus_texts)}")

# Build claim list
claims: List[Dict[str, Any]] = []
for ex in data:
    claims.append({
        "id": ex.get("id"),
        "claim": ex.get("claim", ""),
        "gold_label": normalize_label(ex.get("label")),
        "source": (ex.get("meta") or {}).get("source")
    })
print(f"Claims to evaluate: {len(claims)}")

Loaded 21231 unified examples
Unique evidence texts: 10997
Claims to evaluate: 21231


## Embedder and Reranking

In [7]:
# Load embedding model (E5) and reranker (BGE)
# E5 expects prefixes: "passage: ..." for documents, "query: ..." for queries.

embedder = SentenceTransformer(str(EMB_MODEL_PATH), device=DEVICE)
reranker = FlagReranker(str(RERANK_MODEL_PATH), use_fp16=(DEVICE=="cuda"))

print("Embedding model and Reranker loaded.")

Embedding model and Reranker loaded.


In [8]:
# Embed all evidence texts as passages and build a FAISS IP index (cosine via normalized vectors).

BATCH_SIZE = 128

def e5_encode_passages(texts: List[str]) -> np.ndarray:
    """Encode passages with 'passage: ' prefix and L2 normalize."""
    prefixed = [f"passage: {t}" for t in texts]
    emb = embedder.encode(prefixed, batch_size=BATCH_SIZE, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)
    return emb.astype("float32")

corpus_emb = e5_encode_passages(corpus_texts)
dim = corpus_emb.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product on normalized vectors == cosine similarity
index.add(corpus_emb)

print("FAISS index built:", index.ntotal, "vectors")

Batches:   0%|          | 0/86 [00:00<?, ?it/s]

FAISS index built: 10997 vectors


In [9]:
# Retrieval (FAISS) + Rerank (BGE) + DeepSeek verification

def e5_encode_query(q: str) -> np.ndarray:
    """Encode a single query with 'query: ' prefix and L2 normalize."""
    q_emb = embedder.encode([f"query: {q}"], convert_to_numpy=True, normalize_embeddings=True)
    return q_emb.astype("float32")

def retrieve(claim_text: str, top_k: int = 20) -> Tuple[List[int], List[float]]:
    """Return top_k indices and scores from FAISS."""
    if index.ntotal == 0:
        return [], []
    q = e5_encode_query(claim_text)
    k = min(top_k, index.ntotal)
    D, I = index.search(q, k)
    return I[0].tolist(), D[0].tolist()

def rerank(claim_text: str, doc_indices: List[int], top_k: int = 5) -> List[int]:
    """Rerank retrieved docs with BGE reranker and return top indices (subset)."""
    if not doc_indices:
        return []
    pairs = [(claim_text, corpus_texts[i]) for i in doc_indices]
    scores = reranker.compute_score(pairs, normalize=True)
    order = np.argsort(scores)[::-1]  # descending
    order = order[: min(top_k, len(order))]
    return [doc_indices[i] for i in order]

# DeepSeek API client (OpenAI-compatible)
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)

SYSTEM_VERIFY = (
    "You are an evidence-grounded verifier. "
    "Given a medical/scientific claim and a list of evidence passages, "
    "decide whether the evidence SUPPORTS, REFUTES, or is NEUTRAL w.r.t. the claim. "
    "Be conservative; if evidence is weak or mixed, output NEUTRAL. "
    "Return strictly JSON: {\"label\": \"Support|Refute|Neutral\", \"rationale\": \"...\", \"citations\": [indices] }"
)

def call_deepseek_verify(
    claim_text: str,
    evidence_list: List[str],
    model: str = "deepseek-chat",
    *,
    temperature: float | None = None,
    top_p: float | None = None,
    max_tokens: int | None = None,
    seed: int | None = None,
    extra_messages: Optional[List[Dict[str, str]]] = None,
) -> Dict[str, Any]:
    """Call DeepSeek(OpenAI-compatible) for verification and return a JSON dict."""
    ev_block = "\n".join([f"[{i}] {e}" for i, e in enumerate(evidence_list)]) or "[0] (no evidence)"
    user_msg = f"Claim:\n{claim_text}\n\nEvidence list:\n{ev_block}"

    sys_msg = {"role": "system", "content": SYSTEM_VERIFY}
    base_msgs = [sys_msg]
    if extra_messages:
        base_msgs.extend(extra_messages)
    base_msgs.append({"role": "user", "content": user_msg})

    gen_kwargs = {
        "model": model,
        "messages": base_msgs,
        "response_format": {"type": "json_object"},
    }
    if temperature is not None:
        gen_kwargs["temperature"] = float(temperature)
    else:
        gen_kwargs["temperature"] = 0.0
    if top_p is not None:
        gen_kwargs["top_p"] = float(top_p)
    if max_tokens is not None:
        gen_kwargs["max_tokens"] = int(max_tokens)
    if seed is not None:
        gen_kwargs["seed"] = int(seed)

    try:
        resp = client.chat.completions.create(**gen_kwargs)
        content = resp.choices[0].message.content
        obj = json.loads(content)
        if not isinstance(obj, dict) or "label" not in obj:
            obj = {"label": "Neutral", "rationale": "unexpected response format", "raw": content}
    except Exception as e:
        obj = {"label": "Neutral", "rationale": f"error: {e}", "citations": []}
    return obj

def parse_label(pred: Any) -> str:
    """Map model label to {Support, Refute, Neutral} with a safe fallback."""
    if isinstance(pred, dict):
        pred = pred.get("label", None)
    if not isinstance(pred, str):
        return "Neutral"
    t = pred.strip().lower()
    if t.startswith("support"):
        return "Support"
    if t.startswith("refute"):
        return "Refute"
    if t.startswith("neutral"):
        return "Neutral"
    return "Neutral"

In [10]:
# Quick smoke test on a single claim

TOP_K = 20
TOP_K_RERANK = 5

# Find the first claim text that is non-empty
test_idx = next((i for i, c in enumerate(claims) if c["claim"] and isinstance(c["claim"], str)), 0)
sample = claims[test_idx]

print("Sample claim:", sample["claim"])
ret_idx, ret_sc = retrieve(sample["claim"], top_k=TOP_K)
reranked_idx = rerank(sample["claim"], ret_idx, top_k=TOP_K_RERANK)
ev_texts = [corpus_texts[i] for i in reranked_idx]

print(f"Retrieved: {len(ret_idx)}  | Reranked used: {len(reranked_idx)}")
pred = call_deepseek_verify(sample["claim"], ev_texts, model="deepseek-chat")

print("Prediction:", pred)

Sample claim: Results on the use of hydroxychloroquine as a treatment for Covid-19 have shown no significant differences in health outcomes between the control group and patients who received the experimental drug. 


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Retrieved: 20  | Reranked used: 5
Prediction: {'label': 'Support', 'rationale': "The claim states that hydroxychloroquine shows no significant differences in health outcomes compared to control groups. Evidence [1] provides multiple randomized controlled trials (e.g., from Spain, Brazil, U.S./Canada, and the UK) that consistently report no clinical benefit, no improvement in symptoms, and no significant differences in outcomes such as death rates or recovery times, aligning with the claim. Evidence [3] notes benefits but without statistical significance, and evidence [4] mentions mixed results but includes studies without benefit, which does not contradict the claim. Evidence [0] and [2] discuss potential effectiveness or side effects but do not directly refute the claim's focus on health outcomes from controlled studies. Overall, the strongest evidence from randomized trials supports the claim, with no substantial refuting evidence.", 'citations': [1]}


## Unsupervised Framework for RAG Detection

- oracle
- probability

In [11]:
from typing import List, Dict, Any, Tuple
import math, random
import numpy as np

# Lightweight similarity / “NLI” proxy
_STOP = {"the","a","an","of","and","to","in","for","on","with","at","by",
         "is","are","was","were","be","been","or","that","this","it","as"}
def _norm_tokens(s: str) -> List[str]:
    return [t for t in ''.join(c.lower() if c.isalnum() else ' ' for c in s).split() if t not in _STOP]

def jaccard_overlap(a: str, b: str) -> float:
    A, B = set(_norm_tokens(a)), set(_norm_tokens(b))
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def default_nli_scorer(claim: str, passage: str) -> float:
    """Score claim-passage pair with DeBERTa MNLI model, return entailment probability"""
    inputs = tokenizer(passage, claim, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    # [contradiction, neutral, entailment]
    entailment_prob = float(probs[2])
    return entailment_prob

def _sigmoid(x): return 1/(1+math.exp(-x))

TOP_K = globals().get("TOP_K", 20)
TOP_K_RERANK = globals().get("TOP_K_RERANK", 5)
G_PROBE_RUNS = 6
W_R  = (0.6, 0.4)  # coverage@K, support_density
W_Re = (1.0,)      # delta_support_rank
W_G  = (0.6, 0.4)  # entropy, flip_rate
ALPHA_R = ALPHA_Re = ALPHA_G = 1.0
T_R = T_Re = T_G = 0.8
MU_R = MU_Re = MU_G = 0.0
random.seed(42)

Some weights of the model checkpoint at /root/autodl-tmp/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [12]:
from dataclasses import dataclass
import re
import math
import json
import numpy as np
import itertools
import random
from typing import List, Dict, Any, Optional, Tuple

# ---------- Retrieval fallback (FAISS -> NumPy cosine) ----------
try:
    import faiss  # type: ignore
    _HAS_FAISS = True
    print("[diag] Using FAISS")
except Exception:
    _HAS_FAISS = False
    print("[diag] FAISS not available, falling back to NumPy cosine")

def _normalize_rows(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True) + eps
    return x / n

def build_index(embeddings: np.ndarray, use_faiss: Optional[bool] = None):
    emb = embeddings.astype(np.float32, copy=False)
    if use_faiss is None:
        use_faiss = _HAS_FAISS
    if use_faiss:
        idx = faiss.IndexFlatIP(emb.shape[1])
        faiss.normalize_L2(emb)
        idx.add(emb)
        return ("faiss", idx)
    else:
        return ("numpy", _normalize_rows(emb))

def search_index(index_handle, query_vec: np.ndarray, top_k: int = 20):
    """Robust top-k search on FAISS/NumPy backend with empty-guard."""
    kind, idx = index_handle
    q = query_vec.astype(np.float32, copy=False)
    if q.ndim == 1:
        q = q[None, :]

    if top_k <= 0:
        return np.array([], dtype=int), np.array([], dtype=np.float32)

    if kind == "faiss":
        # Guard: empty index
        if getattr(idx, "ntotal", 0) == 0:
            return np.array([], dtype=int), np.array([], dtype=np.float32)
        faiss.normalize_L2(q)
        D, I = idx.search(q, top_k)
        return (I[0] if len(I) else np.array([], dtype=int),
                D[0] if len(D) else np.array([], dtype=np.float32))
    else:
        # NumPy cosine with pre-normalized matrix
        if idx.shape[0] == 0:
            return np.array([], dtype=int), np.array([], dtype=np.float32)
        qn = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-12)
        scores = (idx @ qn.T).squeeze(-1)  # [N]
        k = min(top_k, scores.shape[0])
        if k == 0:
            return np.array([], dtype=int), np.array([], dtype=np.float32)
        part = np.argpartition(-scores, k - 1)[:k]
        order = part[np.argsort(-scores[part])]
        return order, scores[order]

# ---------- Label parsing (robust) ----------
_ALLOWED = {"support": "Support", "refute": "Refute", "neutral": "Neutral"}
_LABEL_RE = re.compile(
    r'"?(label|prediction|pred|y_pred|answer)"?\s*[:=]\s*"?([A-Za-z]+)"?', re.I
)

def _parse_label_from_text(text: str) -> str:
    """Parse Support/Refute/Neutral from JSON-ish or free-form text."""
    m = _LABEL_RE.search(text)
    if m:
        val = m.group(2).lower()
        for k in _ALLOWED:
            if k in val:
                return _ALLOWED[k]
    tl = text.lower()
    if any(x in tl for x in ["support", "entail", "true"]):   return "Support"
    if any(x in tl for x in ["refute", "contrad", "false"]):  return "Refute"
    return "Neutral"

def _predict_label(claim: str, ctx_texts: List[str]) -> str:
    """Call your verifier LLM and robustly parse its label."""
    out = call_deepseek_verify(claim, ctx_texts, model="deepseek-chat")
    if isinstance(out, dict):
        for k in ("label", "prediction", "pred", "y_pred", "answer"):
            if k in out and isinstance(out[k], str):
                val = out[k].lower()
                for kk in _ALLOWED:
                    if kk in val:
                        return _ALLOWED[kk]
        text = json.dumps(out, ensure_ascii=False)
    else:
        text = str(out)
    return _parse_label_from_text(text)

# ---------- Semantic coverage & ranking features ----------
# You can pass either an embedding model or a cross-encoder scorer.
# If neither is provided, fallback to Jaccard + NLI scorer.
def _semantic_similarity(a: str, b: str,
                         emb_model: Optional["SentenceTransformer"]=None) -> float:
    """Return cosine similarity in [~0,1] if emb_model is provided; else 0."""
    if emb_model is None:
        return 0.0
    vecs = emb_model.encode([a, b], normalize_embeddings=True, convert_to_numpy=True)
    return float(np.dot(vecs[0], vecs[1]))

def _coverage_at_k(claim: str,
                   top_texts: List[str],
                   evid_texts: List[str],
                   thr: float = 0.5,
                   emb_model: Optional["SentenceTransformer"]=None) -> float:
    """
    Return a continuous coverage score instead of binary.
      - With gold: max semantic match between any gold and top_k text (cosine/Jaccard).
      - Without gold: max NLI score between claim and top_k text.
    Expected range roughly in [0,1] if scorers are normalized.
    """
    if not top_texts:
        return 0.0

    if evid_texts:
        mx = 0.0
        for e in evid_texts:
            for t in top_texts:
                if emb_model is not None:
                    mx = max(mx, _semantic_similarity(e, t, emb_model))
                else:
                    mx = max(mx, jaccard_overlap(e, t))   # expected ~[0,1]
        return float(mx)

    scores = [default_nli_scorer(claim, t) for t in top_texts]
    return float(max(scores)) if scores else 0.0

def _support_density(claim: str, top_texts: List[str]) -> float:
    """Average NLI support score over top texts (expect ~[0,1])."""
    if not top_texts:
        return 0.0
    return float(np.mean([default_nli_scorer(claim, t) for t in top_texts]))

def _delta_support_rank(claim: str, cand_idxs: List[int], corpus_texts: List[str]) -> float:
    """
    Relative position of the best-NLI-support passage within candidate list.
    0.0 = best at front, 1.0 = worst at bottom.
    """
    if not cand_idxs:
        return 1.0
    scores = [(i, default_nli_scorer(claim, corpus_texts[i])) for i in cand_idxs]
    scores.sort(key=lambda x: x[1], reverse=True)
    best_pid = scores[0][0]
    cur_rank = cand_idxs.index(best_pid)
    return cur_rank / max(1, len(cand_idxs) - 1)

# rank correlation (Spearman rho) without SciPy
def _spearman_rho(order_a: List[int], order_b: List[int]) -> float:
    """Spearman rho on the intersection of lists; conservative when overlap is small."""
    if not order_a or not order_b:
        return 0.0
    n = min(len(order_a), len(order_b))
    a = order_a[:n]; b = order_b[:n]
    rank_a = {pid: r for r, pid in enumerate(a)}
    rank_b = {pid: r for r, pid in enumerate(b)}
    common = [pid for pid in a if pid in rank_b][:n]
    if len(common) < 2:
        return 0.0
    da = np.array([rank_a[p] for p in common], dtype=np.float32)
    db = np.array([rank_b[p] for p in common], dtype=np.float32)
    cov = np.cov(da, db, bias=True)[0, 1]
    std = (da.std() * db.std()) + 1e-12
    return float(cov / std)

# ---------- G-probe: randomized prompt/decoding sampling ----------
_G_PROMPT_TWEAKS = [
    "",  # original
    "Please answer strictly with one of: Support, Refute, Neutral.",
    "Be concise. Output only one token: Support / Refute / Neutral.",
    "If the evidence is insufficient, prefer Neutral.",
]
_G_TEMPS  = [0.2, 0.5, 0.8]
_G_TOPPS  = [0.9, 0.95]

def _g_probe_labels(claim: str, X_base: List[str], runs: int = 6, debug: bool = False) -> List[str]:
    """
    Diversified probing:
      - prompt tweaks × (temperature, top_p)
      - plus lightweight context variants (first-k, odd/even, reversed, drop-1)
    Falls back to context/prompt diversity even if the model ignores sampling params.
    """
    import itertools, random

    # 1) Generate several evidence context variants (independent of temperature/top_p)
    if not X_base:
        ctx_variants = [[]]
    else:
        k = min(5, len(X_base))
        xb = X_base[:k]
        ctx_variants = [
            xb,                      # first k
            xb[: max(1, k - 1)],     # drop-last
            xb[::2],                 # odd positions
            xb[1::2] or xb[:1],      # even positions (fallback to 1 item if empty)
            list(reversed(xb)),      # reversed order
        ]

    # 2) Prompt variants (slightly stronger instructions)
    prompt_variants = [
        "",
        "Please answer strictly with one of: Support, Refute, Neutral.",
        "Be concise. Output exactly one token: Support / Refute / Neutral.",
        "If evidence is insufficient, choose Neutral.",
        "Decide and return only: Support, Refute, or Neutral.",
    ]

    # 3) Combine (prompt, ctx_variant, temp, top_p), shuffle and sample up to `runs`
    temps  = _G_TEMPS if '_G_TEMPS' in globals() else [0.2, 0.6, 0.9]
    topps  = _G_TOPPS if '_G_TOPPS' in globals() else [0.8, 0.9, 0.95]
    combos = list(itertools.product(prompt_variants, range(len(ctx_variants)), temps, topps))
    random.shuffle(combos)
    combos = combos[: max(1, runs)]

    labels: List[str] = []
    max_toks = PARAMS.get("MAX_TOKENS", 64)

    for pt, ci, t, p in combos:
        ctx = ctx_variants[ci]
        out = call_deepseek_verify(
            claim + (" " + pt if pt else ""),
            ctx,
            model="deepseek-chat",
            temperature=t,
            top_p=p,
            max_tokens=max_toks,
        )
        # Prefer extracting label from JSON; otherwise fallback to heuristic parsing
        if isinstance(out, dict) and isinstance(out.get("label"), str):
            lab = out["label"].strip().lower()
        else:
            lab = str(out).strip().lower()

        if ("support" in lab) or ("entail" in lab) or ("true" in lab):
            labels.append("Support")
        elif ("refute" in lab) or ("contrad" in lab) or ("false" in lab):
            labels.append("Refute")
        elif "neutral" in lab:
            labels.append("Neutral")
        else:
            labels.append("Neutral")

    if debug:
        print(f"\n[G-probe Debug] Claim: {claim[:60]}...")
        used = [(pt[:22] + ("…" if len(pt) > 22 else ""), ci, t, p) for (pt, ci, t, p) in combos]
        print("  combos_used (prompt_stub, ctx_id, temp, top_p):", used)
        print("  labels:", labels)

    return labels or ["Neutral"]

def _entropy_and_flip(labels: List[str]) -> Tuple[float, float, float]:
    """
    ent: prediction entropy over {S,R,N}; flip: 1 - majority share;
    js: JS divergence between first-half vs second-half label distributions.
    """
    if not labels:
        return 0.0, 0.0, 0.0
    cntS, cntR, cntN = labels.count("Support"), labels.count("Refute"), labels.count("Neutral")
    tot = max(1, len(labels))
    probs = np.array([cntS, cntR, cntN], dtype=np.float64) / tot
    ent = float(-(probs * (np.log(probs + 1e-9))).sum())
    flip = float(1.0 - probs.max())

    mid = tot // 2 or 1
    to_id = lambda l: 0 if l == "Support" else (1 if l == "Refute" else 2)
    p1 = np.bincount([to_id(l) for l in labels[:mid]], minlength=3) / max(1, mid)
    p2 = np.bincount([to_id(l) for l in labels[mid:]], minlength=3) / max(1, tot - mid)
    m = 0.5 * (p1 + p2)
    js = 0.5 * (
        (p1 * (np.log((p1 + 1e-12) / (m + 1e-12)))).sum()
        + (p2 * (np.log((p2 + 1e-12) / (m + 1e-12)))).sum()
    )
    ent  = float(max(ent, 0.0))
    flip = float(max(flip, 0.0))
    js   = float(max(js, 0.0))
    return ent, flip, js


# ---------- Blame calibrator (weakly-supervised) ----------
_SK_OK = False
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.calibration import CalibratedClassifierCV
    _SK_OK = True
    print("[diag] scikit-learn available: using calibrated LR when fitted")
except Exception:
    _SK_OK = False
    print("[diag] scikit-learn not available: using hand-tuned fallback")

@dataclass
class BlameConfig:
    TOP_K: int = 20
    TOP_K_RERANK: int = 5
    G_PROBE_RUNS: int = 6

    # Hand-tuned weights for fallback logits (used if sklearn not available or not fitted)
    # R: coverage, support_density
    W_R: Tuple[float, float] = (1.2, 1.0)
    # Re: delta_support_rank, (1 - spearman)  <-- include ranking disagreement
    W_Re: Tuple[float, float] = (2.0, 1.0)
    # G: entropy, flip, js
    W_G: Tuple[float, float, float] = (1.0, 1.0, 0.5)

    # Oracle trigger gains
    ALPHA_R: float = 1.2
    ALPHA_Re: float = 1.0
    ALPHA_G: float = 0.8

    # Mean shifts and temperatures for fallback logits
    MU_R: float = 0.0
    MU_Re: float = 0.0
    MU_G: float = 0.0
    T_R: float = 1.0
    T_Re: float = 1.0
    T_G: float = 1.0

    # Centralized thresholds (for reference, not all used here)
    COV_THR_EMB: float = 0.6
    COV_THR_JACC: float = 0.3
    NLI_THR_NO_GOLD: float = 0.5

CFG = BlameConfig()  # Allow external override

def _softmax3(zR: float, zRe: float, zG: float) -> Tuple[float, float, float]:
    """Numerically-stable 3-way softmax to get a proper probability simplex."""
    m = max(zR, zRe, zG)
    eR = math.exp(zR - m)
    eRe = math.exp(zRe - m)
    eG = math.exp(zG - m)
    s = eR + eRe + eG + 1e-12
    return eR / s, eRe / s, eG / s

import numpy as np

class BlameCalibrator:
    """
    If sklearn is available and fitted, use calibrated multinomial LR to output [p_R, p_Re, p_G].
    Otherwise, use hand-tuned fallback logits combined with softmax for a proper distribution.
    """
    def __init__(self):
        self._ok = _SK_OK
        self._clf = None
        self._gamma = float(getattr(CFG, "SHARPEN_GAMMA", 1.5))

    # --- minimal, private post-process: probability sharpening ---
    def _post_process_probs(self, probs: np.ndarray) -> np.ndarray:
        # probs: shape (3,), sum to 1
        if abs(self._gamma - 1.0) <= 1e-6:
            return probs
        q = np.power(np.clip(probs, 1e-8, 1.0), self._gamma)
        return q / q.sum()

    def fit(self, feats: List[Dict[str, Any]]):
        if not self._ok:
            return  # fallback only
        X, y = [], []
        for f in feats:
            if f.get("overall_error", 0) != 1:
                continue
            sR, sRe, sG = f["R"]["s_R"], f["Re"]["s_Re"], f["G"]["s_G"]
            s_sum = sR + sRe + sG
            if s_sum == 1:
                if sR:  y.append(0)
                if sRe: y.append(1)
                if sG:  y.append(2)
                X.append([
                    f["R"]["coverage"], f["R"]["support_density"], sR,
                    f["Re"]["delta_support_rank"], f["Re"]["spearman"], sRe,
                    f["G"]["entropy"], f["G"]["flip_rate"], f["G"]["js_div"], sG
                ])
        if not X:
            return
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.int64)
        base = LogisticRegression(max_iter=300, multi_class="multinomial")
        self._clf = CalibratedClassifierCV(base, method="sigmoid", cv=3).fit(X, y)

    def predict_proba(self, f: Dict[str, Any]) -> Tuple[float, float, float]:
        if (self._clf is not None) and self._ok:
            x = np.array([[
                f["R"]["coverage"], f["R"]["support_density"], f["R"]["s_R"],
                f["Re"]["delta_support_rank"], f["Re"]["spearman"], f["Re"]["s_Re"],
                f["G"]["entropy"], f["G"]["flip_rate"], f["G"]["js_div"], f["G"]["s_G"]
            ]], dtype=np.float32)
            probs = self._clf.predict_proba(x)[0]
            probs = np.array([
                float(probs[0] if probs.shape[0] == 3 else 0.33),
                float(probs[1] if probs.shape[0] == 3 else 0.33),
                float(probs[2] if probs.shape[0] == 3 else 0.33),
            ], dtype=np.float64)
            probs = self._post_process_probs(probs)
            return float(probs[0]), float(probs[1]), float(probs[2])

        # --- hand-tuned fallback: combine logits with softmax ---
        zR  = (CFG.W_R[0]  * f["R"]["coverage"]
             + CFG.W_R[1]  * f["R"]["support_density"]
             + CFG.ALPHA_R * f["R"]["s_R"]) - CFG.MU_R

        rho = max(float(f["Re"]["spearman"]), 0.0)  # include ranking disagreement via (1 - max(spearman, 0.0))
        zRe = (CFG.W_Re[0] * f["Re"]["delta_support_rank"]
             + CFG.W_Re[1] * (1.0 - rho)
             + CFG.ALPHA_Re * f["Re"]["s_Re"]) - CFG.MU_Re

        zG  = (CFG.W_G[0]  * f["G"]["entropy"]
             + CFG.W_G[1]  * f["G"]["flip_rate"]
             + CFG.W_G[2]  * f["G"]["js_div"]
             + CFG.ALPHA_G * f["G"]["s_G"]) - CFG.MU_G

        pR, pRe, pG = _softmax3(zR / CFG.T_R, zRe / CFG.T_Re, zG / CFG.T_G)
        probs = self._post_process_probs(np.array([pR, pRe, pG], dtype=np.float64))
        return float(probs[0]), float(probs[1]), float(probs[2])

CALIBRATOR = BlameCalibrator()  # global instance (optional training outside)

# ---------- Evidence localization (same API) ----------
def _find_evidence_ids(evid_texts: List[str],
                       corpus_texts: List[str],
                       top_m: int = 3,
                       emb_model: Optional["SentenceTransformer"]=None) -> List[int]:
    """
    Map gold evidence texts to likely passage IDs in the corpus using embedding cosine or Jaccard,
    keeping top_m per gold and deduping by best score.
    """
    scored: List[Tuple[int, float]] = []
    for e in evid_texts:
        loc: List[Tuple[int, float]] = []
        for pid, t in enumerate(corpus_texts):
            if emb_model is not None:
                s = _semantic_similarity(e, t, emb_model)
                if s > 0:
                    loc.append((pid, s))
            else:
                s = jaccard_overlap(e, t)
                if s > 0:
                    loc.append((pid, s))
        loc.sort(key=lambda x: x[1], reverse=True)
        scored.extend(loc[:top_m])

    scored.sort(key=lambda x: x[1], reverse=True)
    seen, out = set(), []
    for pid, _ in scored:
        if pid not in seen:
            seen.add(pid)
            out.append(pid)
    return out

[diag] Using FAISS
[diag] scikit-learn available: using calibrated LR when fitted


### run_one

In [13]:
## Max_length
MAX_LEN = (
    getattr(tokenizer, "model_max_length", None)
    or getattr(model.config, "max_position_embeddings", None)
    or 512
)

if MAX_LEN > 4096:
    MAX_LEN = 512
print(f"[INFO] MAX_LEN set to {MAX_LEN}")

id2label = getattr(model.config, "id2label", None)
if id2label:
    id2label = {k: v.upper() for k, v in id2label.items()}
    IDX_E = next((k for k, v in id2label.items() if "ENTAIL" in v), 2)
    IDX_C = next((k for k, v in id2label.items() if "CONTRAD" in v), 0)
    IDX_N = next((k for k, v in id2label.items() if "NEUT" in v), 1)
else:
    IDX_E, IDX_C, IDX_N = 2, 0, 1
print(f"[INFO] Label indices -> E:{IDX_E}  C:{IDX_C}  N:{IDX_N}")

# --- Robust NLI probs for a single pair: returns (pE, pC, pN), truncation flag, token count ---
def nli_probs_and_len(claim: str, passage: str) -> tuple:
    """
    Return (pE, pC, pN, truncated:bool, n_tokens:int).
    Requires you already defined tokenizer/model/MAX_LEN + robust label indices.
    """
    # assumes you defined IDX_E, IDX_C, IDX_N and MAX_LEN as in your scorer
    inputs = tokenizer(
        passage, claim,
        return_tensors="pt",
        truncation=True, padding="max_length", max_length=MAX_LEN
    )
    n_tokens = int(inputs["input_ids"].shape[1])
    # whether we actually hit the max length cap (approx trunc indicator)
    truncated = (n_tokens >= MAX_LEN)

    with torch.no_grad():
        logits = model(**{k: v.to(device) for k, v in inputs.items()}).logits
        probs = torch.softmax(logits, dim=-1)[0].detach().cpu().numpy()

    pE = float(probs[IDX_E]); pC = float(probs[IDX_C]); pN = float(probs[IDX_N])
    return pE, pC, pN, truncated, n_tokens

def _to01(x: float) -> float:
    import numpy as _np
    return float(_np.clip(x, 0.0, 1.0))

[INFO] MAX_LEN set to 512
[INFO] Label indices -> E:2  C:0  N:1


In [14]:
def run_one(sample: Dict[str, Any],
            corpus_texts: List[str],
            emb_model: Optional["SentenceTransformer"]=None,
            debug: bool = False,
            system_name: str = "Baseline") -> Dict[str, Any]:
    """
    Stage-wise diagnosis for a single claim.

    Pipeline:
      1) Base: retrieve -> rerank -> select top-K' passages -> LLM label
      2) R-oracle: inject likely-evidence into candidates -> rerank -> LLM label
      3) Re-oracle: ideal reordering (by NLI) on base candidates -> LLM label
      4) G-probe: diversified decoding to measure instability (entropy/flip/js)

    Returns:
      Flat metrics + nested features, with consistent columns across systems.
    """
    # ----------- unpack sample -----------
    claim = (sample.get("claim") or "").strip()
    gold  = sample.get("gold_label") or sample.get("label")
    evid_texts = [e.get("text", "") for e in sample.get("evidence", []) if isinstance(e, dict)]

    # ---------- Base pipeline ----------
    # External dependencies expected:
    #   retrieve(claim, top_k) -> (idxs, scores) or idxs
    #   rerank(claim, idxs, top_k) -> ordered idxs (best first)
    ret_idx, _ = retrieve(claim, top_k=CFG.TOP_K)
    if isinstance(ret_idx, np.ndarray):
        ret_idx = ret_idx.tolist()

    reranked_idx = rerank(claim, ret_idx, top_k=CFG.TOP_K_RERANK)
    if isinstance(reranked_idx, np.ndarray):
        reranked_idx = reranked_idx.tolist()

    X_base_ids = reranked_idx[:CFG.TOP_K_RERANK]
    X_base = [corpus_texts[i] for i in X_base_ids]
    y_base = _predict_label(claim, X_base)
    e_overall = int(gold is not None and y_base != gold)

    # --- Base top-1 NLI-based confidence & truncation info ---
    conf_top1 = None
    nli_margin_top1 = None
    nli_truncated_top1 = None
    n_tokens_top1 = None
    try:
        if X_base:
            pE, pC, pN, trunc, n_tok = nli_probs_and_len(claim, X_base[0])
            nli_margin_top1 = float(pE - pC)                     # [-1, 1]
            conf_top1       = _to01(0.5 * (pE - pC + 1.0))       # [0, 1]
            nli_truncated_top1 = bool(trunc)
            n_tokens_top1      = int(n_tok)
    except Exception:
        pass

    # ---------- R-oracle ----------
    # If we have gold evidence, map them to passage ids; else, synthesize likely-evidence
    if evid_texts:
        evid_ids = _find_evidence_ids(evid_texts, corpus_texts, top_m=3, emb_model=emb_model)
    else:
        # broader retrieve, score by NLI, take top few as pseudo-evidence
        big_idx, _ = retrieve(claim, top_k=max(CFG.TOP_K, 100))
        if isinstance(big_idx, np.ndarray):
            big_idx = big_idx.tolist()
        scored = [(i, default_nli_scorer(claim, corpus_texts[i])) for i in big_idx]
        scored.sort(key=lambda x: x[1], reverse=True)
        evid_ids = [i for i, _ in scored[:3]]

    # merge candidates (dedup but keep order)
    cand_R = list(dict.fromkeys(list(ret_idx) + list(evid_ids)))
    rerank_R = rerank(claim, cand_R, top_k=CFG.TOP_K_RERANK)
    if isinstance(rerank_R, np.ndarray):
        rerank_R = rerank_R.tolist()
    X_R_ids = rerank_R[:CFG.TOP_K_RERANK]
    X_R = [corpus_texts[i] for i in X_R_ids]
    y_R = _predict_label(claim, X_R)
    s_R = int(e_overall == 1 and gold is not None and y_R == gold)

    # --- R-stage hits & best-evidence ranks (requires evid_ids) ---
    hit_at_5 = hit_at_10 = None
    rank_best_ret = rank_best_rerank = None
    if evid_ids:
        # rank in retrieval list
        ranks_ret = [ret_idx.index(e) for e in evid_ids if e in ret_idx]
        if ranks_ret:
            rank_best_ret = int(min(ranks_ret))
        # rank in reranked list
        ranks_rer = [reranked_idx.index(e) for e in evid_ids if e in reranked_idx]
        if ranks_rer:
            rank_best_rerank = int(min(ranks_rer))
        # recall@k
        hit_at_5  = int(any((e in reranked_idx[:5])  for e in evid_ids))
        hit_at_10 = int(any((e in reranked_idx[:10]) for e in evid_ids))

    # ---------- Re-oracle ----------
    # Ideal sort by NLI score on the SAME base candidate set
    ret_scores = [(i, default_nli_scorer(claim, corpus_texts[i])) for i in ret_idx]
    ideal_sorted = [i for i, _ in sorted(ret_scores, key=lambda x: x[1], reverse=True)]
    X_Re_ids = ideal_sorted[:CFG.TOP_K_RERANK]
    X_Re = [corpus_texts[i] for i in X_Re_ids]
    y_Re = _predict_label(claim, X_Re)
    s_Re = int(e_overall == 1 and gold is not None and y_Re == gold)

    # Spearman between your reranker order and the ideal NLI order
    spearman_val = _spearman_rho(reranked_idx, ideal_sorted)

    # ---------- G-probe ----------
    reranked_texts = [corpus_texts[i] for i in reranked_idx[:CFG.TOP_K_RERANK]]
    g_labels = _g_probe_labels(claim, reranked_texts, runs=CFG.G_PROBE_RUNS, debug=debug)
    H, F, JS = _entropy_and_flip(g_labels)
    # majority label from the probe
    if g_labels:
        # deterministic majority (tie-break: Support > Refute > Neutral)
        counts = {
            "Support": g_labels.count("Support"),
            "Refute":  g_labels.count("Refute"),
            "Neutral": g_labels.count("Neutral")
        }
        g_maj = max(counts.keys(), key=lambda k: (counts[k], {"Support":2, "Refute":1, "Neutral":0}[k]))
    else:
        g_maj = y_base
    s_G = int(e_overall == 1 and gold is not None and g_maj != gold)

    # ---------- Features ----------
    cov   = _coverage_at_k(claim, X_base, evid_texts, emb_model=emb_model)
    dens  = _support_density(claim, X_base)
    d_rank= _delta_support_rank(claim, ret_idx, corpus_texts)

    feature_dict = {
        "overall_error": e_overall,
        "R":  {"coverage": float(cov), "support_density": float(dens), "s_R": int(s_R)},
        "Re": {"delta_support_rank": float(d_rank), "spearman": float(spearman_val), "s_Re": int(s_Re)},
        "G":  {"entropy": float(H), "flip_rate": float(F), "js_div": float(JS), "s_G": int(s_G)},
    }

    # ---------- Probabilities ----------
    p_R, p_Re, p_G = CALIBRATOR.predict_proba(feature_dict)

    # expose calibrator backend if available
    calib_backend = None
    try:
        calib_backend = CALIBRATOR.backend_name()
    except Exception:
        calib_backend = "unknown"

    # ---------- Output (flat + nested) ----------
    out: Dict[str, Any] = {
        "system": system_name,
        "id": sample.get("id"),
        "claim": claim,
        "gold": gold,
        "y_base": y_base,
        "overall_error": e_overall,

        # blame probabilities
        "p_R": float(p_R),
        "p_Re": float(p_Re),
        "p_G": float(p_G),

        # flattened key metrics for CSV
        "cov": float(cov),
        "dens": float(dens),
        "Δrank": float(d_rank),
        "spearman_rho": float(spearman_val),
        "H": float(H),
        "Flip": float(F),
        "JS": float(JS),

        # NLI-based top1 confidence & truncation
        "conf_top1": conf_top1,
        "nli_margin_top1": nli_margin_top1,
        "nli_truncated_top1": nli_truncated_top1,
        "n_tokens_top1": n_tokens_top1,

        # R-stage recalls & best-evidence ranks
        "hit_at_5": hit_at_5,
        "hit_at_10": hit_at_10,
        "rank_of_best_evidence_ret": rank_best_ret,
        "rank_of_best_evidence_rerank": rank_best_rerank,

        # calibrator backend
        "calib_backend": calib_backend,

        # nested features (kept for debugging/ablation)
        "R_features":  feature_dict["R"],
        "Re_features": feature_dict["Re"],
        "G_features":  feature_dict["G"],
    }

    if debug:
        # NLI scores for current base candidates (ret_idx order)
        try:
            ret_nli_scores = [default_nli_scorer(claim, corpus_texts[i]) for i in ret_idx]
        except Exception:
            ret_nli_scores = []
        try:
            ideal_nli_scores = [default_nli_scorer(claim, corpus_texts[i]) for i in ideal_sorted[:CFG.TOP_K_RERANK]]
        except Exception:
            ideal_nli_scores = []

        out["debug"] = {
            "ret_idx": ret_idx,
            "ret_nli_scores": ret_nli_scores,
            "reranked_idx": reranked_idx,
            "reranked_texts": X_base,  # texts given to LLM at base
            "ideal_sorted_idx": ideal_sorted[:CFG.TOP_K_RERANK],
            "ideal_nli_scores": ideal_nli_scores,
            "X_base_ids": X_base_ids,
            "X_R_ids": X_R_ids,
            "X_Re_ids": X_Re_ids,
            "X_base": X_base,
            "X_R": X_R,
            "X_Re": X_Re,
            "y_R": y_R,
            "y_Re": y_Re,
            "g_probe_labels": g_labels,
            "g_probe_majority": g_maj,
            "coverage_components": {
                "has_gold_evidence": bool(evid_texts),
                "num_top_texts": len(X_base),
                "coverage_score": float(cov),
            },
        }

    return out

### parameters

In [15]:
# ======= One-Stop Config & Runner (LOW-MEM edition; drop-in) =======
from dataclasses import dataclass
from typing import Optional, List, Dict, Any
import os, gc, random, numpy as np, pandas as pd
from tqdm import tqdm

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
random.seed(1234); np.random.seed(1234)

PARAMS = {
    # Retrieval & Reranking (more conservative)
    "TOP_K": 32,
    "TOP_K_RERANK": 6,

    # G-probe (more conservative)
    "G_PROBE_RUNS": 6,
    "PROBE_TEMPS": [0.2, 0.6],
    "PROBE_TOPPS": [0.9],
    "MAX_TOKENS": 64,

    # Semantic coverage (memory-friendly)
    "USE_SEMANTIC_COVERAGE": True,
    "EMB_MODEL_PATH": "intfloat/e5-small-v2",
    "COV_THR_SEM": 0.6,
    "COV_THR_JACC": 0.3,
    "NLI_THR_NO_GOLD": 0.5,

    # Probability calibration
    "USE_LEARNED_CALIBRATOR": True,

    # Number of runs & chunking
    "NUM_EXAMPLES": 150,
    "CHUNK_SIZE": 30,            # Chunk size (avoid loading too much at once)
    "BATCH_CSV_INTERVAL": 5,     # Write CSV every N examples
    "STREAM_ONLY": True,         # True: stream directly to CSV, no in-memory rows
                                 # If on-the-fly calibration is needed, will auto-switch to semi-streaming

    # OOM backoff (adaptive)
    "OOM_BACKOFF_TOPK": 16,      # On OOM: reduce TOP_K to this value
    "OOM_BACKOFF_RERANK": 4,     # On OOM: reduce TOP_K_RERANK to this value
    "OOM_BACKOFF_GPROBE": 3,     # On OOM: reduce G_PROBE_RUNS to this value

    "SHARPEN_GAMMA": 1.7,
}

In [16]:
# Let G-probe read MAX_TOKENS from PARAMS
try:
    _G_MAX_TOKENS = int(PARAMS.get("MAX_TOKENS", 64))
except NameError:
    pass

# Apply config values to global CFG (if CFG/_G_TEMPS/_G_TOPPS exist)
try:
    CFG.TOP_K = int(PARAMS["TOP_K"])
    CFG.TOP_K_RERANK = int(PARAMS["TOP_K_RERANK"])
    CFG.G_PROBE_RUNS = int(PARAMS["G_PROBE_RUNS"])
    CFG.SHARPEN_GAMMA = float(PARAMS["SHARPEN_GAMMA"])
except NameError:
    pass
try:
    _G_TEMPS[:] = list(PARAMS["PROBE_TEMPS"])
    _G_TOPPS[:] = list(PARAMS["PROBE_TOPPS"])
except NameError:
    pass

emb_model = None
if PARAMS["USE_SEMANTIC_COVERAGE"]:
    try:
        from sentence_transformers import SentenceTransformer
        _device = "cuda" if ("CUDA_VISIBLE_DEVICES" in os.environ or os.environ.get("FORCE_CUDA","")=="1") else "cpu"
        emb_model = SentenceTransformer(PARAMS["EMB_MODEL_PATH"], device=_device)
        # 降低内存占用
        try:
            emb_model.max_seq_length = min(getattr(emb_model, "max_seq_length", 256), 256)
        except Exception:
            pass
    except Exception as e:
        print(f"Semantic coverage load failed, fallback to Jaccard: {e}")
        emb_model = None

def _oom_backoff():
    """Handle CUDA OOM by dynamically reducing expensive settings"""
    try:
        CFG.TOP_K = min(CFG.TOP_K, PARAMS["OOM_BACKOFF_TOPK"])
        CFG.TOP_K_RERANK = min(CFG.TOP_K_RERANK, PARAMS["OOM_BACKOFF_RERANK"])
        CFG.G_PROBE_RUNS = min(CFG.G_PROBE_RUNS, PARAMS["OOM_BACKOFF_GPROBE"])
        print(f"[OOM] Backoff → TOP_K={CFG.TOP_K}, RERANK={CFG.TOP_K_RERANK}, G_PROBE_RUNS={CFG.G_PROBE_RUNS}")
    except NameError:
        pass

def _clear_cuda():
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except Exception:
        pass
    gc.collect()

def evaluate_samples(
    claims: List[Dict[str, Any]],
    corpus_texts: List[str],
    num_examples: Optional[int] = None,
    debug: bool = False,
    calibrate_on_the_fly: bool = False,
) -> pd.DataFrame:
    """
    Streamed, low-memory evaluation runner.
    - Writes CSV at ./diagnostics.csv
    - Optionally collects minimal features to fit the calibrator on-the-fly.
    """
    N = num_examples if num_examples is not None else int(PARAMS["NUM_EXAMPLES"])
    CHUNK = int(PARAMS["CHUNK_SIZE"])
    BATCH_N = int(PARAMS["BATCH_CSV_INTERVAL"])
    STREAM_ONLY = bool(PARAMS["STREAM_ONLY"])

    if calibrate_on_the_fly and PARAMS.get("USE_LEARNED_CALIBRATOR", True):
        STREAM_ONLY = True
        need_feats = True
    else:
        need_feats = False
        
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    if os.path.exists(out_path):
        os.remove(out_path)
    header_written = False

    total = min(N, len(claims))
    indices = np.random.choice(len(claims), size=total, replace=False).tolist()

    feats_buf: List[Dict[str, Any]] = []  
    ids_buf: List[Any] = []               

    for s in range(0, total, CHUNK):
        idx_block = indices[s:s+CHUNK]
        batch_rows: List[Dict[str, Any]] = []

        for bi, idx in enumerate(tqdm(idx_block, desc=f"Block {s//CHUNK+1}/{(total+CHUNK-1)//CHUNK}")):
            ex = claims[idx]
            try:
                res = None
                try:
                    res = run_one(ex, corpus_texts, emb_model=emb_model, debug=debug)
                except TypeError:
                    res = run_one(ex, corpus_texts, debug=debug)
            except RuntimeError as rexc:
                if "out of memory" in str(rexc).lower():
                    _oom_backoff(); _clear_cuda()
                    try:
                        res = run_one(ex, corpus_texts, emb_model=emb_model, debug=debug)
                    except Exception as e2:
                        print("[Skip after OOM backoff]", e2)
                        continue
                else:
                    print("[Skip error]", rexc)
                    continue
            except Exception as e:
                print("[Skip error]", e)
                continue

            lite = {
                # info
                "id": res.get("id"),
                "gold": res.get("gold"),
                "y_base": res.get("y_base"),
                "err": int(res.get("overall_error", 0)),

                # prob
                "p_R": float(res.get("p_R", 0.0)),
                "p_Re": float(res.get("p_Re", 0.0)),
                "p_G": float(res.get("p_G", 0.0)),

                # R / Re / G metrics
                "cov": float(res.get("cov", res.get("R_features", {}).get("coverage", 0.0))),
                "dens": float(res.get("dens", res.get("R_features", {}).get("support_density", 0.0))),
                "Δrank": float(res.get("Δrank", res.get("Re_features", {}).get("delta_support_rank", 0.0))),
                "spearman_rho": float(res.get("spearman_rho", res.get("Re_features", {}).get("spearman", 0.0))),
                "H": float(res.get("H", res.get("G_features", {}).get("entropy", 0.0))),
                "Flip": float(res.get("Flip", res.get("G_features", {}).get("flip_rate", 0.0))),
                "JS": float(res.get("JS", res.get("G_features", {}).get("js_div", 0.0))),

                # NLI top-1 confi / length
                "conf_top1": res.get("conf_top1"),
                "nli_margin_top1": res.get("nli_margin_top1"),
                "nli_truncated_top1": res.get("nli_truncated_top1"),
                "n_tokens_top1": res.get("n_tokens_top1"),

                # R hitting
                "hit_at_5": res.get("hit_at_5"),
                "hit_at_10": res.get("hit_at_10"),
                "rank_of_best_evidence_ret": res.get("rank_of_best_evidence_ret"),
                "rank_of_best_evidence_rerank": res.get("rank_of_best_evidence_rerank"),

                # calib
                "calib_backend": res.get("calib_backend"),
            }
            batch_rows.append(lite)

            # minimal feats - only calib
            if need_feats:
                feat = {
                    "overall_error": int(res.get("overall_error", 0)),
                    "R": {
                        "coverage": float(res.get("R_features", {}).get("coverage", res.get("cov", 0.0))),
                        "support_density": float(res.get("R_features", {}).get("support_density", res.get("dens", 0.0))),
                        "s_R": int(res.get("R_features", {}).get("s_R", 0)),
                    },
                    "Re": {
                        "delta_support_rank": float(res.get("Re_features", {}).get("delta_support_rank", res.get("Δrank", 0.0))),
                        "spearman": float(res.get("Re_features", {}).get("spearman", res.get("spearman_rho", 0.0))),
                        "s_Re": int(res.get("Re_features", {}).get("s_Re", 0)),
                    },
                    "G": {
                        "entropy": float(res.get("G_features", {}).get("entropy", res.get("H", 0.0))),
                        "flip_rate": float(res.get("G_features", {}).get("flip_rate", res.get("Flip", 0.0))),
                        "js_div": float(res.get("G_features", {}).get("js_div", res.get("JS", 0.0))),
                        "s_G": int(res.get("G_features", {}).get("s_G", 0)),
                    },
                }
                feats_buf.append(feat)
                ids_buf.append(lite["id"])

                lbl = "Skip"
                if feat["overall_error"] == 1:
                    sR, sRe, sG = feat["R"]["s_R"], feat["Re"]["s_Re"], feat["G"]["s_G"]
                    s_sum = sR + sRe + sG
                    if s_sum == 1:
                        lbl = "R" if sR else ("Re" if sRe else "G")
                print(f"[feats] id={lite.get('id')} type={lbl}")

            if len(batch_rows) >= BATCH_N:
                pd.DataFrame(batch_rows).to_csv(out_path, mode="a", index=False, header=not header_written)
                header_written = True
                batch_rows.clear()
                _clear_cuda()

        if batch_rows:
            pd.DataFrame(batch_rows).to_csv(out_path, mode="a", index=False, header=not header_written)
            header_written = True
            batch_rows.clear()
            _clear_cuda()

    # Optinal : Calib
    if calibrate_on_the_fly and PARAMS.get("USE_LEARNED_CALIBRATOR", True) and feats_buf:
        try:
            CALIBRATOR.fit(feats_buf)
            print("Using calibrator_fit！")
            p_list = [CALIBRATOR.predict_proba(f) for f in feats_buf]  # [(pR,pRe,pG), ...]
            df_base = pd.read_csv(out_path)
            df_probs = pd.DataFrame({
                "id": ids_buf,
                "p_R": [float(p[0]) for p in p_list],
                "p_Re": [float(p[1]) for p in p_list],
                "p_G": [float(p[2]) for p in p_list],
            })
            df_merged = df_base.drop(columns=["p_R","p_Re","p_G"], errors="ignore").merge(df_probs, on="id", how="left")
            df_merged.to_csv(out_path, index=False)
        except Exception as e:
            print(f"calibrate_on_the_fly({e})")

    try:
        return pd.read_csv(out_path)
    except Exception as e:
        print("cannot read CSV: ", e)
        return pd.DataFrame()

No sentence-transformers model found with name intfloat/e5-small-v2. Creating a new one with mean pooling.


Semantic coverage load failed, fallback to Jaccard: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.


### Running Functions

In [17]:
# -- Cell 3: Run with centralized PARAMS --
df = evaluate_samples(claims, corpus_texts, num_examples=PARAMS["NUM_EXAMPLES"])
df

Block 5/5: 100%|██████████| 30/30 [33:26<00:00, 66.87s/it]


,id,gold,y_base,err,p_R,p_Re,p_G,cov,dens,Δrank,...,JS,conf_top1,nli_margin_top1,nli_truncated_top1,n_tokens_top1,hit_at_5,hit_at_10,rank_of_best_evidence_ret,rank_of_best_evidence_rerank,calib_backend
0,4657,Support,Neutral,1,0.081697,0.649200,0.269103,0.003179,0.001464,0.193548,...,0.0,0.499799,-0.000401,True,512,0,0,6.0,NaN,unknown
1,720,Neutral,Neutral,0,0.544996,0.389536,0.065468,0.917974,0.311245,0.548387,...,0.0,0.306121,-0.387757,True,512,1,1,17.0,1.0,unknown
2,1104,Support,Neutral,1,0.175466,0.777982,0.046552,0.351490,0.062793,0.838710,...,0.0,0.500310,0.000619,True,512,0,0,26.0,NaN,unknown
3,4223,Neutral,Support,1,0.594464,0.212571,0.192964,0.544846,0.096289,0.032258,...,0.0,0.499435,-0.001130,True,512,1,1,1.0,1.0,unknown
4,6452,Neutral,Neutral,0,0.206911,0.617573,0.175516,0.065163,0.031508,0.419355,...,0.0,0.552594,0.105189,True,512,0,0,NaN,NaN,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,7313,Support,Neutral,1,0.110897,0.531606,0.357497,0.012667,0.004449,0.032258,...,0.0,0.295445,-0.409111,True,512,0,0,1.0,NaN,unknown
146,16770,Support,Support,0,0.489061,0.408247,0.102692,0.656625,0.252549,0.258065,...,0.0,0.691664,0.383328,True,512,1,1,1.0,2.0,unknown
147,3037,Support,Neutral,1,0.070289,0.893410,0.036300,0.892457,0.169586,0.967742,...,0.0,0.501086,0.002172,True,512,1,1,16.0,4.0,unknown
148,12173,Support,Support,0,0.399997,0.498943,0.101060,0.569807,0.233391,0.032258,...,0.0,0.518081,0.036163,True,512,0,0,NaN,NaN,unknown
